In [3]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

fsize = 20
plt.rcParams.update({"font.size": fsize})
%config InlineBackend.figure_format = 'retina'

In [4]:
def load_evidence(fn):
    df = pd.read_json(fn)
    # Unwrap the 'source' column (contains a dictionary) into separate columns
    source_df = df['source'].apply(pd.Series)
    # Combine the original DataFrame with the unwrapped columns
    df = pd.concat([df.drop(columns=['source']), source_df], axis=1)
    return df

In [5]:
f1_fn = "../data/adipose_Emont2022/evidence_human/evidence.json"
# f2_fn = "../data/adipose_Vijay2019/evidence_human/evidence.json"

df = load_evidence(f1_fn)

In [6]:
from rapidfuzz.fuzz import partial_ratio

def is_fuzzy_contained(variable, sentence, threshold=70):
    """
    Check if the variable is fuzzily contained in the sentence.
    
    Args:
        variable (str): The string to check for.
        sentence (str): The sentence to check within.
        threshold (int): The similarity threshold (0-100).
                         Higher threshold means stricter match.
    
    Returns:
        bool: True if the variable is fuzzily contained, False otherwise.
    """
    # Ensure inputs are strings
    variable = str(variable).lower().strip()
    sentence = str(sentence).lower().strip()
    
    # Calculate the similarity score
    score = partial_ratio(variable, sentence)
    
    # Check if the score meets the threshold
    return score >= threshold

In [7]:
df["ct_in_source"]   = df.apply(lambda x: is_fuzzy_contained(x["cell_type"], x["source_rationale"], 70), axis=1)
df["gene_in_source"] = df.apply(lambda x: is_fuzzy_contained(x["gene"],      x["source_rationale"], 70), axis=1)

In [8]:
df.query("source_type == 'text'")["gene_in_source"].value_counts()

True    17
Name: gene_in_source, dtype: int64

In [9]:
df.query("source_type == 'text'")["ct_in_source"].value_counts()

True    17
Name: ct_in_source, dtype: int64